In [1]:
import pandas as pd

train = pd.read_csv("../data/train.csv", index_col='id')

train.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [2]:
y = train["Churn"]
X = train.drop(columns=["Churn"])
# Getting info about categorical data

cat_cols = [col for col in X.columns if X[col].dtype == 'object']
num_cols = [col for col in X.columns if X[col].dtype == 'float64' or X[col].dtype == 'int64']

print(cat_cols)
print(num_cols)

['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV



In [4]:
le = LabelEncoder()

y = le.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=1)

print(X_train.shape)
print(X_test.shape)

(475355, 19)
(118839, 19)


In [ ]:
#feature engineering
def new_features(df):
    df["AverageCharges"] = df["TotalCharges"]/(df["tenure"]+1)
    df["ServiceSupport"] = (df[["OnlineSecurity", "TechSupport", "DeviceProtection","OnlineBackup"]] == 'Yes').sum(axis=1)
    df["StreamService"] = (df[["StreamingTV", "StreamingMovies"]] == 'Yes').sum(axis=1)


    df['trusted'] = ((df["Contract"] == 'One year') | (df["Contract"] == 'Two year')).astype(int) + ((df["PaymentMethod"] == 'Credit card (automatic)') | (df["PaymentMethod"] == 'Bank transfer (automatic)')).astype(int)

    return df

X_train = new_features(X_train)
X_test = new_features(X_test)

In [7]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AverageCharges', 'ServiceSupport', 'trusted', 'StreamService']
print(num_cols)

['tenure', 'MonthlyCharges', 'TotalCharges', 'AverageCharges', 'ServiceSupport', 'trusted', 'StreamService']


In [8]:
ord_cat = ['Partner','Dependents', 'PhoneService','PaperlessBilling']

nom_cat = ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection','TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

In [9]:
# Building Pipelines

num_pipe = Pipeline(steps = [("num_impute", SimpleImputer(strategy='mean')),
                             ("num_scaling", StandardScaler())])

ord_cat_pipe = Pipeline(steps = [("ord_impute", SimpleImputer(strategy='most_frequent')),
                                 ("ord_encode", OrdinalEncoder())])

nom_cat_pipe = Pipeline(steps = [("nom_impute", SimpleImputer(strategy='most_frequent')),
                                 ("nom_encode", OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))])

preprocessing = ColumnTransformer(transformers=[('num_trf', num_pipe, num_cols),
                                                ('ord_trf',ord_cat_pipe, ord_cat),
                                                ('nom_trf', nom_cat_pipe, nom_cat)],
                                                remainder='passthrough')



In [10]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(penalty='l2')

lr_pipe = Pipeline(steps=[('preprocessing', preprocessing),
                          ('lr_model', lr)])

lr_pipe.fit(X_train, y_train)

y_pred = lr_pipe.predict_proba(X_test)[:,1]

In [11]:
threshold = [0.1, 0.15, 0.2, 0.25, 0.3]

for t in threshold:
    y_prob = (y_pred > t).astype(int)
    print(f'Classification Report for threshold = {t} \n', classification_report(y_test, y_prob))
    print(roc_auc_score(y_test, y_prob))


Classification Report for threshold = 0.1 
               precision    recall  f1-score   support

           0       0.98      0.68      0.80     92140
           1       0.46      0.95      0.62     26699

    accuracy                           0.74    118839
   macro avg       0.72      0.81      0.71    118839
weighted avg       0.86      0.74      0.76    118839

0.8134232924421986
Classification Report for threshold = 0.15 
               precision    recall  f1-score   support

           0       0.97      0.73      0.83     92140
           1       0.50      0.92      0.65     26699

    accuracy                           0.77    118839
   macro avg       0.73      0.83      0.74    118839
weighted avg       0.86      0.77      0.79    118839

0.8259321643296519
Classification Report for threshold = 0.2 
               precision    recall  f1-score   support

           0       0.96      0.77      0.86     92140
           1       0.53      0.89      0.66     26699

    accurac

In [11]:
test = pd.read_csv("../data/test.csv", index_col='id')

test = new_features(test)
test.head()



,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,AverageCharges,ServiceSupport,StreamService,trusted
id,,,,,,,,,,,,,,,,,,,,,
594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,...,Yes,Two year,Yes,Electronic check,115.55,8061.50,110.431507,6,2,1
594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,...,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50,18.562500,1,0,2
594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,...,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55,48.734615,3,0,1
594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,...,Yes,Two year,No,Credit card (automatic),84.10,6457.15,89.682639,5,2,2
594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,...,Yes,Month-to-month,No,Electronic check,90.35,1233.65,77.103125,2,2,0


In [12]:
X = new_features(X)

In [18]:
lr_pipe.fit(X,y)
lr_pred = lr_pipe.predict_proba(test)[:,1]

In [50]:
lr_final = pd.DataFrame({'id' : test.index, 'Churn': lr_pred})

lr_final.to_csv("../submissions/lr_pred.csv", index=False)

In [27]:
#RandomForest model
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500, 
    class_weight='balanced',
    max_depth=12,
    max_features=0.5,
    max_samples=0.6,
    n_jobs=-1,
    random_state=1,
    max_leaf_nodes=128
    )

rf_pipe = Pipeline(steps=[('preprocessing', preprocessing),
                          ('rf_model', rf)])

rf_pipe.fit(X_train, y_train)

rf_prob = rf_pipe.predict_proba(X_test)[:,1]


In [28]:
threshold = [ 0.4, 0.45, 0.5,0.55,  0.6]

for t in threshold:
    rf_pred = (rf_prob > t).astype(int)
    print(f'Classification Report for threshold = {t} \n', classification_report(y_test, rf_pred))
    print(roc_auc_score(y_test, rf_pred))


Classification Report for threshold = 0.4 
               precision    recall  f1-score   support

           0       0.97      0.74      0.84     92140
           1       0.51      0.92      0.65     26699

    accuracy                           0.78    118839
   macro avg       0.74      0.83      0.75    118839
weighted avg       0.87      0.78      0.80    118839

0.8304038970232857
Classification Report for threshold = 0.45 
               precision    recall  f1-score   support

           0       0.96      0.77      0.86     92140
           1       0.53      0.90      0.67     26699

    accuracy                           0.80    118839
   macro avg       0.75      0.83      0.76    118839
weighted avg       0.87      0.80      0.81    118839

0.8343662942933918
Classification Report for threshold = 0.5 
               precision    recall  f1-score   support

           0       0.96      0.80      0.87     92140
           1       0.55      0.87      0.68     26699

    accurac

In [26]:
params = {
    'rf_model__n_estimators' : [300, 500, 800, 1000],
    'rf_model__max_depth' : [6, 7, 8, 10, 11, 12],
    'rf_model__max_features' : [0.5, 0.6, 0.8, 1], 
    'rf_model__max_samples' : [0.5, 0.6, 0.8, 1],
    'rf_model__max_leaf_nodes' : [24, 32, 48, 64, 128]
}
search = RandomizedSearchCV(rf_pipe, param_distributions=params, n_iter = 10, cv=3, scoring='roc_auc', n_jobs=-1,random_state=1)

search.fit(X_train, y_train)

best_params = search.best_params_
best_score = search.best_score_

print(best_params)
print(best_score)

{'rf_model__n_estimators': 500, 'rf_model__max_samples': 0.6, 'rf_model__max_leaf_nodes': 128, 'rf_model__max_features': 0.5, 'rf_model__max_depth': 12}
0.9118080755956214


In [29]:
rf_pipe.fit(X,y)
rf_pred_final = rf_pipe.predict_proba(test)[:,1]

# rf_final = pd.DataFrame({'id' : test.index, 'Churn': rf_pred_final})

# rf_final.to_csv("../submissions/rf_pred.csv", index=False)

c:\venvs\myenv\lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [46]:
# Getting weighted average of Logistic Regression and Random Forest

weights = [0.1, 0.2, 0.3, 0.4, 0.5]
threshold = [ 0.2, 0.3, 0.4,  0.5, 0.6, 0.7]

for w in weights:
    weighted_prob = w*y_pred + (1-w)*rf_pred
    print(weighted_prob[:5])
    for t in threshold:
        weighted_avg = (weighted_prob > t).astype(int)
        print(f'Classification Report for threshold = {t} and {w} \n', classification_report(y_test, weighted_avg))
        print(roc_auc_score(y_test, weighted_avg))
    

[1.14209178e-05 6.09610347e-03 8.02962916e-03 4.01415249e-04
 1.15590704e-03]
Classification Report for threshold = 0.2 and 0.1 
               precision    recall  f1-score   support

           0       0.94      0.84      0.89     92140
           1       0.60      0.82      0.69     26699

    accuracy                           0.84    118839
   macro avg       0.77      0.83      0.79    118839
weighted avg       0.86      0.84      0.84    118839

0.8296891408357728
Classification Report for threshold = 0.3 and 0.1 
               precision    recall  f1-score   support

           0       0.94      0.84      0.89     92140
           1       0.60      0.82      0.69     26699

    accuracy                           0.84    118839
   macro avg       0.77      0.83      0.79    118839
weighted avg       0.86      0.84      0.84    118839

0.8296891408357728
Classification Report for threshold = 0.4 and 0.1 
               precision    recall  f1-score   support

           0       

In [21]:
final_weighted_prob = 0.2 * lr_pred + 0.8 *rf_pred_final

weighted_final = pd.DataFrame({'id' : test.index, 'Churn': final_weighted_prob})

weighted_final.to_csv("../submissions/lr_rf_weighted_pred_20.csv", index=False)

In [34]:
#Building Gradient Boost Model
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter = 1200,
    max_depth = 6,
    max_leaf_nodes=48,
    max_features= 0.5,
    l2_regularization=0.3,
    random_state=1,
    early_stopping=True
)

hgb_pipe = Pipeline(steps=[("preprocessing", preprocessing),
                           ("hgb_model", hgb)])

hgb_pipe.fit(X_train, y_train)
hgb_prob = hgb_pipe.predict_proba(X_test)[:,1]



In [35]:
for t in [0.1, 0.2, 0.3, 0.4]:
    hgb_pred = (hgb_prob > t).astype(int)
    print(f'Classification Report for threshold = {t} \n', classification_report(y_test, hgb_pred))
    print(roc_auc_score(y_test, hgb_pred))

Classification Report for threshold = 0.1 
               precision    recall  f1-score   support

           0       0.98      0.68      0.81     92140
           1       0.47      0.95      0.63     26699

    accuracy                           0.74    118839
   macro avg       0.72      0.82      0.72    118839
weighted avg       0.86      0.74      0.77    118839

0.8183758637328818
Classification Report for threshold = 0.2 
               precision    recall  f1-score   support

           0       0.96      0.78      0.86     92140
           1       0.54      0.89      0.68     26699

    accuracy                           0.81    118839
   macro avg       0.75      0.84      0.77    118839
weighted avg       0.87      0.81      0.82    118839

0.8386582128594953
Classification Report for threshold = 0.3 
               precision    recall  f1-score   support

           0       0.94      0.84      0.89     92140
           1       0.60      0.83      0.70     26699

    accuracy

In [32]:
hgb_params = {
    'hgb_model__max_iter' : [500, 800, 1000, 1200],
    'hgb_model__max_depth' : [6, 7, 8, 10],
    'hgb_model__max_leaf_nodes' : [32,48,64], 
    'hgb_model__learning_rate' : [0.01, 0.025, 0.05, 0.1],
    'hgb_model__l2_regularization' : [0.05, 0.1, 0.2, 0.25, 0.3]
}
hgb_search = RandomizedSearchCV(hgb_pipe, param_distributions=hgb_params, n_iter = 10, cv=3, scoring='roc_auc', n_jobs=-1,random_state=1)

hgb_search.fit(X_train, y_train)

hgb_best_params = hgb_search.best_params_
hgb_best_score = hgb_search.best_score_

In [33]:
hgb_best_params = hgb_search.best_params_
hgb_best_score = hgb_search.best_score_

print(hgb_best_params)
print(hgb_best_score)

{'hgb_model__max_leaf_nodes': 48, 'hgb_model__max_iter': 1200, 'hgb_model__max_depth': 6, 'hgb_model__learning_rate': 0.05, 'hgb_model__l2_regularization': 0.3}
0.9159675176248346


In [36]:
hgb_pipe.fit(X,y)

hgb_final_pred = hgb_pipe.predict_proba(test)[:,1]

hgb_final_df = pd.DataFrame({'id' : test.index, 'Churn': hgb_final_pred})

hgb_final_df.to_csv("../submissions/hgb_pred.csv", index=False)

In [36]:
#Weighted average of HGB and LR

hgb_lr_avg = 0.1 * lr_pred + 0.9 * hgb_final_pred

hgb_lr_df = pd.DataFrame({'id' : test.index, 'Churn': hgb_lr_avg})

hgb_lr_df.to_csv("../submissions/hgb_lr_pred.csv", index=False)

In [1]:
# Implementing XGBoost
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---

In [17]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators = 1200,
    max_depth = 6,
    colsample_bytree = 0.7,
    colsample_bynode = 0.6,
    colsample_bylevel = 0.7,
    learning_rate = 0.02,
    reg_lambda = 0.2,
    reg_alpha = 0.2
)

xgb_pipe = Pipeline(steps=[("preprocessing", preprocessing),
                           ("xgb_model", xgb)])

xgb_pipe.fit(X_train, y_train)

xgb_prob = xgb_pipe.predict_proba(X_test)[:,1]

In [18]:
for t in [0.1, 0.2, 0.3, 0.4, 0.5]:
    xgb_pred = (xgb_prob > t).astype(int)
    print(f'Classification Report for threshold = {t} \n', classification_report(y_test, xgb_pred))
    print(roc_auc_score(y_test, xgb_pred))
    

Classification Report for threshold = 0.1 
               precision    recall  f1-score   support

           0       0.98      0.68      0.81     92140
           1       0.47      0.95      0.63     26699

    accuracy                           0.74    118839
   macro avg       0.72      0.82      0.72    118839
weighted avg       0.86      0.74      0.77    118839

0.8180450534365242
Classification Report for threshold = 0.2 
               precision    recall  f1-score   support

           0       0.96      0.78      0.86     92140
           1       0.54      0.89      0.68     26699

    accuracy                           0.81    118839
   macro avg       0.75      0.84      0.77    118839
weighted avg       0.87      0.81      0.82    118839

0.8384481716531902
Classification Report for threshold = 0.3 
               precision    recall  f1-score   support

           0       0.94      0.84      0.89     92140
           1       0.60      0.83      0.70     26699

    accuracy

In [16]:
xgb_params = {
    'xgb_model__n_estimators' :[300, 500, 800, 1000, 1200],
    'xgb_model__learning_rate' : [0.01, 0.02, 0.05, 0.1],
    'xgb_model__max_depth' : [6, 7, 8, 10],
    'xgb_model__colsample_bylevel' : [0.5, 0.6, 0.7, 0.8, 0.9],
    'xgb_model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8, 0.9],
    'xgb_model__colsample_bynode' : [0.5, 0.6, 0.7, 0.8, 0.9], 
    'xgb_model__reg_alpha' : [0.1, 0.2, 0.25, 0.3],
    'xgb_model__reg_lambda' : [0.1, 0.2, 0.25, 0.3]
    
}
xgb_search = RandomizedSearchCV(xgb_pipe, param_distributions=xgb_params, n_iter = 10, cv=3, scoring='roc_auc', n_jobs=-1,random_state=1)

xgb_search.fit(X_train, y_train)

xgb_best_params = xgb_search.best_params_
xgb_best_score = xgb_search.best_score_

print(xgb_best_params)
print(xgb_best_score)

{'xgb_model__reg_lambda': 0.2, 'xgb_model__reg_alpha': 0.2, 'xgb_model__n_estimators': 1200, 'xgb_model__max_depth': 6, 'xgb_model__learning_rate': 0.02, 'xgb_model__colsample_bytree': 0.7, 'xgb_model__colsample_bynode': 0.6, 'xgb_model__colsample_bylevel': 0.7}
0.9162684384510827


In [19]:
xgb_pipe.fit(X,y)

xgb_final_pred = xgb_pipe.predict_proba(test)[:,1]

xgb_final_df = pd.DataFrame({'id' : test.index, 'Churn': xgb_final_pred})

xgb_final_df.to_csv("../submissions/xgb_pred.csv", index=False)

In [13]:
print(type(y))

<class 'numpy.ndarray'>


In [14]:
y = pd.DataFrame({'Churn' : y})


In [15]:
from sklearn.model_selection import KFold
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier

kf = KFold(n_splits=5, shuffle=True, random_state=1)

oof_hgb = np.zeros(len(X))
test_hgb = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    hgb_fold = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter = 1200,
        max_depth = 6,
        max_leaf_nodes=48,
        max_features= 0.5,
        l2_regularization=0.3,
        random_state=1,
        early_stopping=True,
        validation_fraction=0.1
    )

    hgb_fold_pipe = Pipeline(steps=[("preprocessing", preprocessing),
                                    ("hgb_fold_model", hgb_fold)])
    
    hgb_fold_pipe.fit(X_tr, y_tr)

    oof_hgb[val_idx] = hgb_fold_pipe.predict_proba(X_val)[:,1]
    test_hgb += hgb_fold_pipe.predict_proba(test)[:,1]
    



c:\venvs\myenv\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\venvs\myenv\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\venvs\myenv\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\venvs\myenv\lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = col

In [16]:
test_hgb = test_hgb/ kf.n_splits
hgbfold_df = pd.DataFrame({'id' : test.index, 'Churn': test_hgb})

hgbfold_df.to_csv("../submissions/hgbfold_pred.csv", index=False)

In [16]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
import numpy as np


kf = KFold(n_splits=5, shuffle=True, random_state=1)

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    xgb_fold = XGBClassifier(
        n_estimators = 1200,
        max_depth = 6,
        colsample_bytree = 0.7,
        colsample_bynode = 0.6,
        colsample_bylevel = 0.7,
        learning_rate = 0.02,
        reg_lambda = 0.2,
        reg_alpha = 0.2
    )

    xgb_fold_pipe = Pipeline(steps=[("preprocessing", preprocessing),
                                    ("xgb_fold_model", xgb_fold)])
    
    xgb_fold_pipe.fit(X_tr, y_tr)

    oof_xgb[val_idx] = xgb_fold_pipe.predict_proba(X_val)[:,1]
    test_xgb += xgb_fold_pipe.predict_proba(test)[:,1]
    



In [17]:
test_xgb = test_xgb/ kf.n_splits
xgbfold_df = pd.DataFrame({'id' : test.index, 'Churn': test_xgb})

xgbfold_df.to_csv("../submissions/xgbfold_pred.csv", index=False)

In [19]:
hgb_xgb = 0.9 * test_hgb + 0.1 * test_xgb 
xgbhgb_df = pd.DataFrame({'id' : test.index, 'Churn': hgb_xgb})

xgbhgb_df.to_csv("../submissions/xgb-hgb_pred.csv", index=False)
    